# 29 - Train-only differentiable patch matcher

**Colab GPU, expected T4 sufficient; run02 on the SAME runtime after reset. No NBIS.**
This is the prerequisite for the risk-aware restoration pilot, not a new fingerprint recognition paper.
No downloaded biometric embeddings, external training data, validation image reads, or test access.
The existing frozen library versions remain required. GPU name may differ and is recorded.

A small128-D CNN (GroupNorm/PReLU, no dropout) learns from12 identities x2 byte-distinct impressions per batch:
six fixed epochs x200 updates, FP32, Adam0.0002,192px patches, fixed seed20261110.
Main TRAIN identities/byte-duplicate connected groups are partitioned into fit, eight calibration groups/DB,
and eight gate groups/DB. Calibration and gate use different impressions, not the aligned restoration target.
Same-DB comparisons exclude duplicate-linked different identities.

The LAST checkpoint after the fixed budget is used, not whichever epoch has the best gate.
Calibration sets one fixed1% cosine-score threshold per DB on ORIGINAL patches.
A pragmatic gate requires macro clean AUC>=0.80, dry AUC>=0.70, clean TAR>=0.25 and dry TAR>=0.10.
These are pilot signal-quality criteria, not statistical guarantees; achieved gate FMR is reported separately.
If it fails,29 persists its diagnostics but **30 MUST STOP**. Do not reroll, lower the gate or secretly use another checkpoint.

Expected T4 fit:24 matcher patches/step; no diffusion, no full-image backpropagation.
CUDA peak allocated/reserved memory and elapsed time are recorded. On OOM, do not silently change batch/precision;
restart the SAME configuration on a GPU with more VRAM (L4/A100). GPU changes alter runtime provenance.
Safe content-addressed epoch checkpoints resume; interrupted epochs restart, completed fitting skips.


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import shutil
import sys
import time
import types
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Subset
from google.colab import drive
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU kernel. A T4 is sufficient.')
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
M1_DIR = PROJECT_ROOT / 'experiments' / 'm1_shared_curriculum_v1_seed20261110' / 'dd1427cfad6621c577bf'
SEED = 20261110
RUN_SEEDS = {'dry_l1_seed20261110': 20261110}
RUN_NAMES = tuple(RUN_SEEDS)
M2_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / '92579202157d85254b7f'
SPECIALIZATION_ROOT = PROJECT_ROOT / 'experiments' / 'm5_dry_specialization_v1'
LAMBDA_GRADIENT = 0.1
IDENTITY_FRACTION = 0.1

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

def verify_files(directory, hashes):
    for name, expected in hashes.items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Frozen artifact changed: {directory / name}')
print('GPU:', torch.cuda.get_device_name(0), '| One full-train dry-only L1 diagnostic; no NBIS or test evaluation.')
MATCHER_ROOT = PROJECT_ROOT / 'experiments' / 'risk_patch_matcher_v1'
MATCHER_SETTINGS = {'epochs': 6, 'steps_per_epoch': 200, 'identities_per_batch': 12,
    'learning_rate': 0.0002, 'embedding_dim': 128, 'held_groups_per_db': 8,
    'quality_gate': {'clean_auc': 0.80, 'dry_auc': 0.70, 'clean_tar': 0.25, 'dry_tar': 0.10}}


In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
verify_files(SPLIT_DIR, split_protocol['file_sha256'])
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
m0_config = json.loads((M0_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_config = json.loads((M1_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_completion = json.loads((M1_DIR / 'completion.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
m0_sha, m1_sha = (file_digest(directory / 'configuration.json') for directory in (M0_DIR, M1_DIR))
verify_files(M0_DIR, m0_config['source_file_sha256'])
verify_files(M1_DIR, m1_config['source_file_sha256'])
verify_files(M1_DIR, m1_completion['file_sha256'])
if (m1_completion['status'] != 'shared_curriculum_control_complete' or m1_completion['configuration_sha256'] != m1_sha
        or m1_config['m0_configuration_sha256'] != m0_sha):
    raise RuntimeError('Frozen completed M1 provenance mismatch.')
if any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json') for config in (m0_config, sim_config, pair_config)):
    raise RuntimeError('Split provenance differs across notebooks.')
if (m0_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or m1_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']
        or m0_config['simulator_module_sha256'] != sim_config['module_sha256']):
    raise RuntimeError('Frozen simulator/normalization provenance mismatch.')
for config in (m0_config, m1_config):
    if (str(torch.__version__) != config['torch_version'] or str(torch.version.cuda) != config['torch_cuda']
            or np.__version__ != config['numpy_version'] or Image.__version__ != config['pillow_version']):
        raise RuntimeError('Library versions differ from frozen training.')
if Image.__version__ != pair_config['pillow_version'] or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']:
    raise RuntimeError('Normalization/simulator library versions differ.')
for name in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv'):
    if file_digest(M1_DIR / name) != file_digest(M0_DIR / name):
        raise RuntimeError(f'M1 model/source differs from M0: {name}')
SETTINGS = dict(m1_config['settings'])
EPOCHS = len(m1_config['epoch_conditions'])
if (EPOCHS != sum(SETTINGS['phase_lengths']) or EPOCHS != m0_config['epochs']
        or SETTINGS['base_channels'] != m0_config['base_channels'] or SETTINGS['patch_size'] != m0_config['patch_size']
        or SETTINGS['batch_size'] != m0_config['batch_size'] or SETTINGS['learning_rate'] != m0_config['learning_rate']):
    raise RuntimeError('Budget/architecture/optimizer differ from the frozen baseline.')
train_sources = pd.read_csv(M0_DIR / 'train_sources.csv', keep_default_na=False)
validation_sources = pd.read_csv(M0_DIR / 'validation_sources.csv', keep_default_na=False)
for split_name, frame in (('train', train_sources), ('validation', validation_sources)):
    allowed = locked[locked['split'].eq(split_name)].set_index('relative_path')
    if frame.empty or frame['relative_path'].duplicated().any() or frame.groupby(['dataset', 'db']).ngroups != 6 or set(frame['dataset']) != {'FVC2000', 'FVC2002'}:
        raise RuntimeError(f'Invalid source-only {split_name} data.')
    for record in frame.to_dict('records'):
        relative = record['relative_path']
        if (relative not in allowed.index or allowed.loc[relative, 'sha256'] != record['sha256']
                or allowed.loc[relative, 'identity_key'] != record['identity_key']):
            raise RuntimeError(f'Split/source mismatch: {relative}')
        path = DATA_ROOT / relative
        if split_name == 'train' and (not path.is_file() or file_digest(path) != record['sha256']):
            raise RuntimeError(f'Image missing/changed: {path}. Restore with notebook 02.')
if (set(train_sources['identity_key']) & set(validation_sources['identity_key'])
        or set(train_sources['sha256']) & set(validation_sources['sha256'])
        or not validation_sources.groupby(['dataset', 'db']).size().eq(12).all()):
    raise RuntimeError('Train/validation overlap or changed validation coverage.')
training = load_module('m2_original_model_v1', M0_DIR / 'restoration_baseline.py')
simulator = load_module('m2_original_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
curriculum_data = load_module('m2_frozen_task_data_v1', M1_DIR / 'curriculum_data.py')
FROZEN_ENGINE_SOURCE = (M1_DIR / 'training_helpers.py.txt').read_bytes().decode('utf-8')
SOURCE_COUNT = len(train_sources)
all_conditions = [('identity', 'none')] + [(family, severity)
    for family in ('dry_like', 'wet_like', 'generic') for severity in ('mild', 'moderate', 'severe')]
print('Verified unchanged sources:', SOURCE_COUNT, 'train /', len(validation_sources), 'validation.')

allowed_train_paths = frozenset(train_sources.relative_path)


In [ ]:
MATCHER_SOURCE = "import math\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nclass PatchMatcher(nn.Module):\n    def __init__(self):\n        super().__init__()\n        layers, incoming = [], 1\n        for outgoing in (16, 32, 64, 128):\n            layers.extend((nn.Conv2d(incoming, outgoing, 3, stride=2, padding=1),\n                nn.GroupNorm(4, outgoing), nn.PReLU(outgoing)))\n            incoming = outgoing\n        self.features = nn.Sequential(*layers)\n        self.embedding = nn.Linear(128 * 4 * 4, 128)\n\n    def forward(self, images):\n        if images.ndim != 4 or images.shape[1] != 1 or min(images.shape[-2:]) < 16:\n            raise ValueError('Matcher requires grayscale N1HW patches of at least16 pixels.')\n        features = self.features(images)\n        height, width = features.shape[-2:]\n        # Match adaptive-average bins without its nondeterministic CUDA backward.\n        bins = [features[:, :, row * height // 4:((row + 1) * height + 3) // 4,\n            column * width // 4:((column + 1) * width + 3) // 4].mean(dim=(-2, -1))\n            for row in range(4) for column in range(4)]\n        features = torch.stack(bins, dim=-1).flatten(1)\n        return F.normalize(self.embedding(features), dim=1, eps=1e-8)\n\ndef identity_groups(frame):\n    parents = {key: key for key in frame.identity_key.unique()}\n    def root(key):\n        while parents[key] != key:\n            parents[key] = parents[parents[key]]\n            key = parents[key]\n        return key\n    for _, rows in frame.groupby('sha256', sort=True):\n        roots = sorted({root(key) for key in rows.identity_key})\n        for key in roots[1:]:\n            parents[key] = roots[0]\n    return {key: root(key) for key in parents}\n\ndef inner_partition(frame, held_groups=8):\n    groups = identity_groups(frame)\n    result = frame.copy()\n    result['connected_group'] = result.identity_key.map(groups)\n    group_dbs = result.assign(database=result.dataset + '/' + result.db).groupby('connected_group').database.nunique()\n    if not group_dbs.eq(1).all():\n        raise RuntimeError('Cross-database duplicate group: explicitly redesign inner stratification.')\n    result['matcher_split'] = 'fit'\n    for (dataset, db), rows in result.groupby(['dataset', 'db'], sort=True):\n        keys = sorted(rows.connected_group.unique(),\n            key=lambda key: training.stable_seed(f'{SEED}|risk_matcher_split|{key}'))\n        if len(keys) < 2 * held_groups + 4:\n            raise RuntimeError(f'Insufficient train-only groups in {dataset}/{db} for fixed matcher calibration/gate.')\n        for label, selected in (('calibration', keys[:held_groups]),\n                                ('gate', keys[held_groups:2 * held_groups])):\n            result.loc[result.connected_group.isin(selected), 'matcher_split'] = label\n    if result.groupby('connected_group').matcher_split.nunique().max() != 1:\n        raise RuntimeError('Inner duplicate/identity leakage.')\n    return result\n\ndef sample_patches(records, epoch, conditions):\n    paths = [row['relative_path'] for row in records]\n    if any(path not in allowed_train_paths for path in paths):\n        raise RuntimeError('Matcher may read main TRAIN images only.')\n    dataset = training.FingerprintPairs(records, DATA_ROOT, simulator,\n        pair_config['source_dpi'], SEED, patch_size=SETTINGS['patch_size'], training=True)\n    dataset.epoch, dataset.conditions = epoch, list(conditions)\n    if not dataset.conditions:\n        raise ValueError('Explicit matcher view conditions required.')\n    return torch.stack([dataset[index]['input'] for index in range(len(records))])\n\ndef supervised_contrastive(embeddings, labels, temperature=0.1):\n    if embeddings.ndim != 2 or len(labels) != len(embeddings) or temperature <= 0:\n        raise ValueError('Aligned embeddings/labels and positive temperature required.')\n    diagonal = torch.eye(len(labels), dtype=torch.bool, device=embeddings.device)\n    positives = labels[:, None].eq(labels[None, :]) & ~diagonal\n    if not positives.any(dim=1).all() or not (~labels[:, None].eq(labels[None, :])).any():\n        raise ValueError('Every anchor needs a positive and at least one different identity.')\n    logits = (embeddings @ embeddings.T / temperature).masked_fill(diagonal, -torch.inf)\n    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)\n    return -(log_prob.masked_fill(~positives, 0).sum(1) / positives.sum(1)).mean()\n\ndef matcher_batch(partition, epoch, step, identities=12):\n    fit = partition[partition.matcher_split.eq('fit')]\n    keys = sorted(fit.sort_values('identity_key').drop_duplicates('connected_group').identity_key)\n    if len(keys) < identities:\n        raise RuntimeError('Too few fit identities for fixed P-K matcher batches.')\n    rng = np.random.default_rng(training.stable_seed(f'{SEED}|matcher_batch|{epoch}|{step}'))\n    chosen = rng.choice(keys, identities, replace=False)\n    records, labels = [], []\n    for label, key in enumerate(chosen):\n        rows = fit[fit.identity_key.eq(key)].drop_duplicates('sha256').to_dict('records')\n        if len(rows) < 2:\n            raise RuntimeError(f'Matcher fit identity needs two byte-distinct impressions: {key}')\n        indices = rng.choice(len(rows), 2, replace=False)\n        records.extend(rows[int(index)] for index in indices)\n        labels.extend([label, label])\n    patches = sample_patches(records, epoch * MATCHER_SETTINGS['steps_per_epoch'] + step,\n        [('identity', 'none'), ('dry_like', 'mild'), ('dry_like', 'moderate'), ('dry_like', 'severe')])\n    return patches.to(DEVICE), torch.tensor(labels, device=DEVICE)\n\n@torch.inference_mode()\ndef verification_scores(model, partition, split):\n    frame = partition[partition.matcher_split.eq(split)]\n    if split not in ('calibration', 'gate') or frame.empty:\n        raise ValueError('Explicit nonempty train-only calibration/gate partition required.')\n    galleries, probes = [], []\n    for key, rows in frame.groupby('identity_key', sort=True):\n        records = rows.sort_values('relative_path').drop_duplicates('sha256').to_dict('records')\n        if len(records) < 3:\n            raise RuntimeError(f'Independent-impression gate requires three byte-distinct sources: {key}')\n        galleries.append(records[0])\n        probes.extend(records[1:3])\n    model.eval()\n    def embed(records, conditions):\n        chunks = []\n        for start in range(0, len(records), 24):\n            images = sample_patches(records[start:start + 24], 0, conditions).to(DEVICE)\n            chunks.append(model(images).cpu())\n        return torch.cat(chunks)\n    gallery_features = embed(galleries, [('identity', 'none')])\n    result = []\n    for family, severity in [('identity', 'none')] + [('dry_like', level) for level in ('mild', 'moderate', 'severe')]:\n        features = embed(probes, [(family, severity)])\n        scores = (features @ gallery_features.T).numpy()\n        for i, probe in enumerate(probes):\n            for j, gallery in enumerate(galleries):\n                if (probe['dataset'], probe['db']) != (gallery['dataset'], gallery['db']):\n                    continue\n                label = int(probe['identity_key'] == gallery['identity_key'])\n                if not label and probe['connected_group'] == gallery['connected_group']:\n                    continue\n                result.append(dict(dataset=probe['dataset'], db=probe['db'],\n                    condition=f'{family}/{severity}', probe=probe['relative_path'],\n                    gallery=gallery['relative_path'], label=label, score=float(scores[i, j])))\n    return pd.DataFrame(result)\n\ndef continuous_threshold(scores, target=0.01):\n    values = np.asarray(scores, dtype=np.float64)\n    if values.ndim != 1 or not len(values) or not np.isfinite(values).all() or not 0 <= target < 1:\n        raise ValueError('Finite nonempty scores and target in[0,1) required.')\n    allowed = math.floor(target * len(values) + 1e-12)\n    return float(np.nextafter(np.sort(values)[::-1][allowed], np.inf))\n\ndef gate_analysis(calibration, gate):\n    thresholds, rows = {}, []\n    for (dataset, db), frame in calibration.groupby(['dataset', 'db'], sort=True):\n        impostors = frame[frame.condition.eq('identity/none') & frame.label.eq(0)].score.to_numpy()\n        if len(impostors) < 100:\n            raise RuntimeError('Calibration resolution cannot support the predefined1% surrogate ceiling.')\n        thresholds[(dataset, db)] = continuous_threshold(impostors)\n    for (dataset, db, condition), frame in gate.groupby(['dataset', 'db', 'condition'], sort=True):\n        genuine, impostor = (frame[frame.label.eq(label)].score.to_numpy() for label in (1, 0))\n        if not len(genuine) or not len(impostor) or (dataset, db) not in thresholds:\n            raise RuntimeError('Incomplete independent-impression matcher gate pairs.')\n        threshold = thresholds[(dataset, db)]\n        auc = float(np.mean((genuine[:, None] > impostor).astype(float)\n                            + 0.5 * (genuine[:, None] == impostor)))\n        rows.append(dict(dataset=dataset, db=db, condition=condition, auc=auc,\n            tar=float(np.mean(genuine >= threshold)), fmr=float(np.mean(impostor >= threshold)),\n            threshold=threshold, genuine_pairs=len(genuine), impostor_pairs=len(impostor)))\n    metrics = pd.DataFrame(rows)\n    expected = {(dataset, db, condition) for dataset, db in thresholds for condition in\n        ('identity/none', 'dry_like/mild', 'dry_like/moderate', 'dry_like/severe')}\n    if set(zip(metrics.dataset, metrics.db, metrics.condition)) != expected or len(thresholds) != 6:\n        raise RuntimeError('Gate needs all four conditions in all six source databases.')\n    clean = metrics[metrics.condition.eq('identity/none')]\n    dry = metrics[~metrics.condition.eq('identity/none')]\n    summary = dict(clean_auc=float(clean.auc.mean()), dry_auc=float(dry.auc.mean()),\n        clean_tar=float(clean.tar.mean()), dry_tar=float(dry.tar.mean()),\n        clean_fmr=float(clean.fmr.mean()), dry_fmr=float(dry.fmr.mean()))\n    limits = MATCHER_SETTINGS['quality_gate']\n    summary['passed'] = all(summary[key] >= value for key, value in limits.items())\n    summary['limits'] = limits\n    summary['interpretation'] = 'Pragmatic train-internal signal-quality gate, not a confidence bound or NBIS accuracy guarantee.'\n    return thresholds, metrics, summary\n"
matcher = types.ModuleType('risk_patch_matcher_v1')
matcher.__dict__.update(globals())
exec(compile(MATCHER_SOURCE, '<risk-patch-matcher>', 'exec'), matcher.__dict__)


In [ ]:
CHECKPOINT_SOURCE = 'def save_state(directory, state):\n    temporary = directory / \'checkpoint.part\'\n    torch.save(state, temporary)\n    digest = file_digest(temporary)\n    filename = f"epoch_{state[\'completed_epochs\']:03d}_{digest[:16]}.pt"\n    target = directory / filename\n    if target.exists() and file_digest(target) != digest:\n        raise RuntimeError(\'Content-addressed checkpoint collision.\')\n    temporary.replace(target)\n    pointer = dict(file=filename, sha256=digest, configuration_sha256=experiment_sha,\n        completed_epochs=state[\'completed_epochs\'])\n    write_verified(directory / \'latest.json\', json_bytes(pointer))\n    return pointer\n\ndef load_state(directory):\n    pointer = json.loads((directory / \'latest.json\').read_text(encoding=\'utf-8\'))\n    if (Path(pointer[\'file\']).name != pointer[\'file\']\n            or pointer[\'configuration_sha256\'] != experiment_sha\n            or file_digest(directory / pointer[\'file\']) != pointer[\'sha256\']):\n        raise RuntimeError(\'Checkpoint pointer/integrity mismatch.\')\n    state = torch.load(directory / pointer[\'file\'], map_location=\'cpu\', weights_only=True)\n    if state[\'configuration_sha256\'] != experiment_sha or state[\'completed_epochs\'] != pointer[\'completed_epochs\']:\n        raise RuntimeError(\'Checkpoint payload mismatch.\')\n    return pointer, state\n'
exec(compile(CHECKPOINT_SOURCE, '<risk-checkpoint-helpers>', 'exec'))


In [ ]:

partition = matcher.inner_partition(train_sources, MATCHER_SETTINGS['held_groups_per_db'])
partition_data = partition.to_csv(index=False, lineterminator='\n').encode()
experiment_config = {'version': 'risk_patch_matcher_v1', 'seed': SEED,
    'settings': MATCHER_SETTINGS, 'patch_size': SETTINGS['patch_size'],
    'split_protocol_sha256': file_digest(SPLIT_DIR / 'protocol.json'),
    'train_sources_sha256': file_digest(M0_DIR / 'train_sources.csv'),
    'model_source_sha256': hashlib.sha256(MATCHER_SOURCE.encode()).hexdigest(),
    'checkpoint_source_sha256': hashlib.sha256(CHECKPOINT_SOURCE.encode()).hexdigest(),
    'partition_sha256': hashlib.sha256(partition_data).hexdigest(),
    'simulator_sha256': file_digest(SIM_DIR / 'fingerprint_degradation.py'),
    'torch_version': str(torch.__version__), 'torch_cuda': str(torch.version.cuda),
    'numpy_version': np.__version__, 'pillow_version': Image.__version__,
    'gpu_name': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
    'selection': 'last fixed-budget checkpoint; no gate/validation epoch selection'}
experiment_sha = hashlib.sha256(json_bytes(experiment_config)).hexdigest()
REPORT_DIR = MATCHER_ROOT / experiment_sha[:20]
REPORT_DIR.mkdir(parents=True, exist_ok=True)
for name, data in {'configuration.json': json_bytes(experiment_config),
        'matcher_source.py': MATCHER_SOURCE.encode(), 'checkpoint_helpers.py.txt': CHECKPOINT_SOURCE.encode(),
        'inner_partition.csv': partition_data}.items():
    path = REPORT_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen matcher configuration changed: {path}')
    if not path.exists():
        write_verified(path, data)
print(partition.groupby(['dataset', 'db', 'matcher_split']).identity_key.nunique().to_string())


In [ ]:

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
model = matcher.PatchMatcher().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=MATCHER_SETTINGS['learning_rate'])
if (REPORT_DIR / 'latest.json').is_file():
    _, state = load_state(REPORT_DIR)
    model.load_state_dict(state['model'], strict=True)
    optimizer.load_state_dict(state['optimizer'])
    torch.set_rng_state(state['torch_rng'])
    torch.cuda.set_rng_state_all(state['cuda_rng'])
    completed, history = state['completed_epochs'], state['history']
else:
    completed, history = 0, []
if not 0 <= completed <= MATCHER_SETTINGS['epochs']:
    raise RuntimeError('Matcher checkpoint exceeds fixed epoch budget.')
if DEVICE.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
for epoch in range(completed, MATCHER_SETTINGS['epochs']):
    started, total = time.monotonic(), 0.0
    model.train()
    for step in range(MATCHER_SETTINGS['steps_per_epoch']):
        images, labels = matcher.matcher_batch(partition, epoch, step, MATCHER_SETTINGS['identities_per_batch'])
        optimizer.zero_grad(set_to_none=True)
        loss = matcher.supervised_contrastive(model(images), labels)
        if not torch.isfinite(loss):
            raise RuntimeError('Nonfinite matcher loss.')
        loss.backward()
        if any(parameter.grad is not None and not torch.isfinite(parameter.grad).all() for parameter in model.parameters()):
            raise RuntimeError('Nonfinite matcher gradient.')
        optimizer.step()
        total += float(loss.detach())
        if (step + 1) % 50 == 0:
            print(f'Matcher epoch{epoch + 1}, step{step + 1}: loss={total / (step + 1):.5f}', flush=True)
    history.append(dict(epoch=epoch + 1, loss=total / MATCHER_SETTINGS['steps_per_epoch'],
        seconds=time.monotonic() - started,
        peak_allocated_bytes=torch.cuda.max_memory_allocated() if DEVICE.type == 'cuda' else 0,
        peak_reserved_bytes=torch.cuda.max_memory_reserved() if DEVICE.type == 'cuda' else 0))
    save_state(REPORT_DIR, dict(configuration_sha256=experiment_sha, completed_epochs=epoch + 1,
        model=model.state_dict(), optimizer=optimizer.state_dict(), history=history,
        torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all()))
    write_csv(REPORT_DIR / 'history.csv', pd.DataFrame(history))
    print('Matcher epoch persisted:', epoch + 1, flush=True)
write_csv(REPORT_DIR / 'history.csv', pd.DataFrame(history))
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
print('Matcher parameters:', sum(parameter.numel() for parameter in model.parameters()))


In [ ]:

calibration = matcher.verification_scores(model, partition, 'calibration')
gate = matcher.verification_scores(model, partition, 'gate')
thresholds, gate_metrics, gate_summary = matcher.gate_analysis(calibration, gate)
write_csv(REPORT_DIR / 'calibration_scores.csv', calibration)
write_csv(REPORT_DIR / 'gate_scores.csv', gate)
write_csv(REPORT_DIR / 'gate_metrics.csv', gate_metrics)
write_verified(REPORT_DIR / 'quality_gate.json', json_bytes(gate_summary))
write_verified(REPORT_DIR / 'thresholds.json', json_bytes(
    {f'{dataset}_{db}': value for (dataset, db), value in thresholds.items()}))
print(gate_metrics.to_string(index=False))
print(json.dumps(gate_summary, indent=2))


In [ ]:

latest = json.loads((REPORT_DIR / 'latest.json').read_text(encoding='utf-8'))
files = ['configuration.json', 'matcher_source.py', 'checkpoint_helpers.py.txt', 'inner_partition.csv',
    'history.csv', 'calibration_scores.csv', 'gate_scores.csv', 'gate_metrics.csv',
    'quality_gate.json', 'thresholds.json', 'latest.json', latest['file']]
completion = dict(status='risk_matcher_fit_complete', configuration_sha256=experiment_sha,
    completed_epochs=MATCHER_SETTINGS['epochs'], quality_gate=gate_summary,
    file_sha256={name: file_digest(REPORT_DIR / name) for name in files},
    checkpoint=latest, main_validation_images_read=False, test_images_read=False)
write_verified(REPORT_DIR / 'completion.json', json_bytes(completion))
write_verified(MATCHER_ROOT / 'completed_experiment.json', json_bytes(dict(directory=str(REPORT_DIR),
    configuration_sha256=experiment_sha, completion_sha256=file_digest(REPORT_DIR / 'completion.json'))))
if not gate_summary['passed']:
    raise RuntimeError('MATCHER QUALITY GATE FAILED: diagnostics saved; STOP before30. No restoration runs authorized by this gate.')
print('RISK MATCHER COMPLETE; QUALITY GATE PASSED:', REPORT_DIR)
print('Run30. Gate is not a guarantee of NBIS transfer or publication novelty.')
